# 🌀 Understanding Hallucinations in LLMs

**Day 4 — Notebook 1 of 4 | Estimated Time: 25 minutes**

---

## What You'll Learn
- What hallucinations are and why they occur in language models
- The main categories of hallucinations (factual, reasoning, attribution)
- How to identify hallucinations through real examples
- Why hallucinations are an inherent property of how LLMs work

---

## Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0"

In [ ]:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai
from google.genai import types
from IPython.display import Markdown

client = genai.Client(api_key=get_api_key())
MODEL_ID = "gemini-2.5-flash"
print("✅ Ready!")

---

## 1. What Is a Hallucination?

A **hallucination** is when an LLM generates content that is:
- **Factually incorrect** — stating false information as fact
- **Fabricated** — inventing citations, names, dates, or events that don't exist
- **Inconsistent** — contradicting itself or the context provided
- **Ungrounded** — making claims not supported by any evidence

### Why do LLMs hallucinate?

LLMs are trained to predict the **next most likely token** given a context. They don't "look up" facts — they generate text that *sounds plausible* based on patterns in training data.

Key causes:
1. **Training data gaps** — the model never saw accurate information on a topic
2. **Knowledge cutoff** — events after the training date are unknown
3. **Conflicting data** — contradictory information in training data
4. **Overgeneralization** — the model extrapolates incorrectly from patterns
5. **Instruction following pressure** — the model "tries to be helpful" even when it doesn't know the answer

---

## 2. Categories of Hallucinations

| Category | Description | Example |
|----------|-------------|----------|
| **Factual** | Wrong facts about the world | Stating a historical date incorrectly |
| **Attribution** | Fake citations, quotes, or sources | Inventing a paper that doesn't exist |
| **Entity** | Wrong names, places, or identities | Misidentifying who wrote a book |
| **Reasoning** | Correct premise, wrong conclusion | Bad math or logical errors |
| **Context** | Ignoring or contradicting provided context | Answering a question the input didn't ask |

---

## 3. Inducing a Factual Hallucination

Let's deliberately ask about a **fictional entity** that doesn't exist and observe the model's behavior.

In [ ]:
# Ask about a completely fictional author to see if the model fabricates a response
response = client.models.generate_content(
    model=MODEL_ID,
    contents="""Tell me about the famous 19th century novelist Bartholomew Quincey-Finch 
    and his celebrated novel 'The Whispering Meridian' published in 1887.""",
)
print(response.text)

**Observe:** Did the model invent details? Or did it correctly say it doesn't know?

Better-calibrated models will refuse or express uncertainty. Older/smaller models often fabricate plausible-sounding but entirely false details.

---

## 4. Citation Hallucinations (Fabricated Sources)

One of the most dangerous hallucinations is **inventing academic papers or sources**.

In [ ]:
# Ask the model to provide academic citations
response = client.models.generate_content(
    model=MODEL_ID,
    contents="""Provide 5 academic papers with full citations (authors, title, journal, year, DOI)
    about the effect of music tempo on workplace productivity.""",
)
print(response.text)

**Warning:** Citations generated by LLMs often look real but have **fake DOIs, wrong authors, or don't exist at all**. Always verify any citation from an LLM using Google Scholar or PubMed.

---

## 5. Reasoning Hallucinations (Confident but Wrong)

LLMs can be **confidently wrong** on arithmetic and logic problems.

In [ ]:
# Classic hallucination trap: the model may get this wrong
response = client.models.generate_content(
    model=MODEL_ID,
    contents="How many times does the letter 'r' appear in the word 'strawberry'?",
)
print(response.text)

In [ ]:
# Let's verify the actual count programmatically
word = "strawberry"
count = word.count('r')
print(f"Actual count of 'r' in '{word}': {count}")
print(f"Letters: {list(word)}")

---

## 6. Context Contradiction Hallucinations

LLMs can sometimes ignore or contradict the context they are given.

In [ ]:
# Provide a context with specific facts, then ask about something NOT in it
context = """
Employee record:
- Name: Sarah Chen
- Department: Engineering
- Start date: March 5, 2021
- Manager: David Park
"""

response = client.models.generate_content(
    model=MODEL_ID,
    contents=f"""Based on the following employee record, what is Sarah's salary?

{context}""",
)
print(response.text)

**Good behavior:** The model should say the salary is not provided in the context.  
**Hallucination:** The model invents a salary figure.

Modern models like Gemini 2.5 Flash are well-calibrated here, but always test your specific use case.

---

## 7. Measuring Confidence vs. Accuracy

A key insight: **LLM confidence does not correlate with accuracy**. The model uses the same fluent, confident tone whether it's right or wrong.

In [ ]:
# Ask for obscure facts that are hard to verify — watch the tone
questions = [
    "What is the boiling point of gold in Kelvin?",
    "What was the exact population of Constantinople in 1450 CE?",
    "How many words did Shakespeare write in total across all his works?",
]

for q in questions:
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=q,
    )
    print(f"Q: {q}")
    print(f"A: {response.text.strip()}")
    print("-" * 60)

---

## 🏋️ Exercise 1: Spot the Hallucination

Ask the model to describe a **real but obscure topic** you know well. Identify any factual errors.

In [ ]:
# Exercise 1: Test hallucination on a topic you know
# TODO: Pick a topic you personally know well (your city, your profession, a hobby)
# Ask Gemini a specific factual question about it
# Evaluate whether the answer is accurate

my_topic = ""  # TODO: Replace with your chosen topic
my_question = ""  # TODO: Write a specific factual question

# response = client.models.generate_content(
#     model=MODEL_ID,
#     contents=my_question,
# )
# print(response.text)
# print()
# print("Is this accurate? What did the model get wrong?")

---

## 🏋️ Exercise 2: Prompt for Self-Uncertainty

Modify a prompt to ask the model to express its **confidence level** and flag anything it's unsure about.

In [ ]:
# Exercise 2: Ask the model to rate its own confidence
# TODO: Add instructions for the model to:
#   1. Answer the question
#   2. Rate its confidence (High / Medium / Low) on each fact
#   3. Flag any details it is uncertain about

question = "What are the key differences between BERT and GPT-3 in terms of architecture and use cases?"

# TODO: Modify the prompt to elicit confidence ratings
# prompt = f"""
# ...
# {question}
# ...
# """

# response = client.models.generate_content(model=MODEL_ID, contents=prompt)
# print(response.text)

---

## Key Takeaways

| Insight | Implication |
|---------|-------------|
| LLMs predict tokens, not facts | Fluency ≠ accuracy |
| Confidence is not calibrated | Never trust an LLM citation without verification |
| Knowledge cutoffs are real | Ask about recent events with caution |
| Context gaps lead to invention | If something isn't in context, the model may fabricate it |
| Better prompts reduce (but don't eliminate) hallucinations | Grounding is the real solution — see Notebook 2 |

---

## 📚 Further Reading

| Resource | Type | Link |
|----------|------|------|
| Survey of Hallucination in LLMs | Paper | [arxiv.org/abs/2311.05232](https://arxiv.org/abs/2311.05232) |
| LLM Hallucinations Explained | Video | [IBM Technology (YouTube)](https://www.youtube.com/watch?v=cfqtFvWOfg0) |
| Gemini Safety & Reliability | Docs | [ai.google.dev/gemini-api/docs](https://ai.google.dev/gemini-api/docs) |

---

**Next up:** [02_Grounding_Techniques.ipynb](./02_Grounding_Techniques.ipynb)